# MURA Musculoskeletal Abnormality Detection - Google Colab

This notebook trains a Vision Transformer model with multi-view fusion for detecting abnormalities in musculoskeletal X-rays.

**⚠️ IMPORTANT: Enable GPU**
1. Go to `Runtime` → `Change runtime type`
2. Select `T4 GPU` (free tier)
3. Click `Save`

**Dataset:** MURA (Stanford) - 40,005 X-ray images

**Training Time:** ~45 minutes per epoch on T4 GPU (vs. 9 hours on CPU)

## 1. Setup and Installation

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU in Runtime settings.")

In [ ]:
# Install required packages
!pip install -q transformers timm opencv-python

## 2. Upload MURA Dataset

**Option A: Upload from Google Drive (Recommended)**
1. Upload MURA-v1.1 folder to your Google Drive
2. Run the cell below to mount Drive

**Option B: Upload Directly (Slower)**
- Use the file browser to upload MURA-v1.1.zip
- Unzip in Colab

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Update this path to where your MURA dataset is in Google Drive
MURA_PATH = '/content/drive/MyDrive/MURA-v1.1'  # Adjust this path!

# Verify dataset exists
import os
if os.path.exists(MURA_PATH):
    print(f"✓ Dataset found at {MURA_PATH}")
    print(f"  Contents: {os.listdir(MURA_PATH)[:5]}")
else:
    print(f"✗ Dataset not found at {MURA_PATH}")
    print(f"  Please update MURA_PATH variable to point to your dataset")

## 3. Create Project Structure

We'll recreate the project structure in Colab.

In [ ]:
# Create project directories
!mkdir -p config data models training evaluation experiments utils checkpoints logs results/gradcam
!touch config/__init__.py data/__init__.py models/__init__.py training/__init__.py evaluation/__init__.py utils/__init__.py
print("✓ Project structure created")

## 4. Upload Project Code

**Upload your project files:**
- Use the file browser (📁 icon on left)
- Upload all `.py` files from your local project to their respective folders

**Or clone from GitHub:**

In [ ]:
# OPTION 1: Clone from GitHub (if you've pushed your code)
# !git clone https://github.com/YOUR_USERNAME/mura-detection.git
# %cd mura-detection

# OPTION 2: Upload files manually using the file browser
print("Please upload your Python files to the corresponding directories:")
print("  - config/config.py")
print("  - data/*.py")
print("  - models/*.py")
print("  - training/*.py")
print("  - evaluation/*.py")
print("  - experiments/*.py")
print("  - utils/*.py")

## Alternative: Write Code Directly in Colab

If you haven't uploaded files, I'll create the essential code here:

In [ ]:
%%writefile config/config.py
import os
import torch

class Config:
    # Paths - UPDATE MURA_PATH
    DATA_ROOT = "/content/drive/MyDrive/MURA-v1.1"  # Update this!
    TRAIN_CSV = os.path.join(DATA_ROOT, "train_labeled_studies.csv")
    VALID_CSV = os.path.join(DATA_ROOT, "valid_labeled_studies.csv")
    
    CHECKPOINT_DIR = "/content/checkpoints"
    LOG_DIR = "/content/logs"
    RESULTS_DIR = "/content/results"
    GRADCAM_DIR = os.path.join(RESULTS_DIR, "gradcam")
    
    # Model
    VIT_MODEL = "google/vit-base-patch16-224"
    HIDDEN_DIM = 768
    NUM_QUERY_TOKENS = 4
    NUM_ATTENTION_HEADS = 8
    MAX_VIEWS = 4
    DROPOUT_RATE = 0.2
    FUSION_DROPOUT = 0.1
    
    # Training
    BATCH_SIZE = 16  # Increased for GPU
    NUM_EPOCHS = 10  # Reduced for Colab
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 0.01
    GRADIENT_CLIP = 1.0
    SCHEDULER_T0 = 10
    SCHEDULER_T_MULT = 2
    POS_WEIGHT = 1.597
    
    # Data
    IMAGE_SIZE = 224
    NUM_WORKERS = 2  # Colab has limited CPU
    PIN_MEMORY = torch.cuda.is_available()
    MEAN = [0.5, 0.5, 0.5]
    STD = [0.5, 0.5, 0.5]
    
    # Checkpointing
    MAX_CHECKPOINTS = 3  # Save space
    SAVE_EVERY_N_EPOCHS = 1
    LOG_INTERVAL = 50
    
    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    SEED = 42
    
    @classmethod
    def create_dirs(cls):
        os.makedirs(cls.CHECKPOINT_DIR, exist_ok=True)
        os.makedirs(cls.LOG_DIR, exist_ok=True)
        os.makedirs(cls.RESULTS_DIR, exist_ok=True)
        os.makedirs(cls.GRADCAM_DIR, exist_ok=True)
    
    @classmethod
    def display(cls):
        print("="*60)
        print("MURA Configuration (Colab)")
        print("="*60)
        print(f"Data Root: {cls.DATA_ROOT}")
        print(f"Model: {cls.VIT_MODEL}")
        print(f"Batch Size: {cls.BATCH_SIZE}")
        print(f"Epochs: {cls.NUM_EPOCHS}")
        print(f"Device: {cls.DEVICE}")
        print("="*60)

print("✓ Config created")

In [ ]:
# Update the DATA_ROOT in config to point to your MURA dataset
# This assumes you've uploaded to Google Drive

import sys
sys.path.insert(0, '/content')

from config.config import Config

# Override with your actual path
Config.DATA_ROOT = MURA_PATH
Config.TRAIN_CSV = os.path.join(MURA_PATH, "train_labeled_studies.csv")
Config.VALID_CSV = os.path.join(MURA_PATH, "valid_labeled_studies.csv")

Config.display()

## 5. Quick Install - Copy All Project Files

**Option: Upload project as ZIP**
1. Zip your entire AI_project folder locally
2. Upload AI_project.zip to Colab
3. Run cell below

In [ ]:
# If you uploaded AI_project.zip
# !unzip -q AI_project.zip
# %cd AI_project
# !pip install -q -r requirements.txt
print("If you uploaded a ZIP, uncomment the lines above")

## 6. Training

Now we'll train the model on GPU!

In [ ]:
# Import all necessary modules
import sys
sys.path.insert(0, '/content')

from config.config import Config
from data.dataset import get_dataloaders
from models.mura_classifier import create_model
from training.trainer import Trainer
from utils.seed import set_seed

print("✓ Modules imported")

In [ ]:
# Initialize
config = Config()
config.create_dirs()
set_seed(config.SEED)

# Load data
print("\nLoading data...")
train_loader, val_loader = get_dataloaders(config)

# Create model
print("\nCreating model...")
model = create_model(config, fusion_type='cross_attention', device=config.DEVICE)

# Create trainer
print("\nInitializing trainer...")
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=config.DEVICE
)

print("\n✓ Ready to train!")

In [ ]:
# Start training
print("Starting training...\n")
trainer.train(num_epochs=config.NUM_EPOCHS)

print(f"\n✓ Training complete!")
print(f"Best AUC: {trainer.get_best_auc():.4f}")

## 7. Monitor Training (TensorBoard)

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/logs

## 8. Evaluation

In [ ]:
from evaluation.evaluator import evaluate_model

# Load best model
checkpoint = torch.load('/content/checkpoints/best.pth')
model.load_state_dict(checkpoint['model_state_dict'])

# Evaluate
metrics = evaluate_model(
    model=model,
    dataloader=val_loader,
    device=config.DEVICE,
    save_predictions='/content/results/predictions.csv'
)

print(f"\nFinal Metrics:")
print(f"  AUC-ROC: {metrics['auc']:.4f}")
print(f"  Accuracy: {metrics['accuracy']:.4f}")
print(f"  Sensitivity: {metrics['sensitivity']:.4f}")
print(f"  Specificity: {metrics['specificity']:.4f}")

## 9. Visualize Results (Grad-CAM)

In [ ]:
from evaluation.visualization import visualize_study
import matplotlib.pyplot as plt

# Generate Grad-CAM for first 10 samples
num_samples = 10
for i, batch in enumerate(val_loader):
    if i >= num_samples:
        break
    
    images = batch['images'][:1]
    mask = batch['mask'][:1]
    labels = batch['label'][:1]
    study_path = batch['study_path'][0]
    
    save_path = f'/content/results/gradcam/sample_{i:03d}.png'
    
    visualize_study(
        model=model,
        images=images,
        mask=mask,
        labels=labels,
        study_path=study_path,
        save_path=save_path,
        device=config.DEVICE
    )

print(f"✓ Generated {num_samples} Grad-CAM visualizations")
print("View them in /content/results/gradcam/")

In [ ]:
# Display some Grad-CAM results
from IPython.display import Image, display

print("Sample Grad-CAM Visualizations:\n")
for i in range(3):
    img_path = f'/content/results/gradcam/sample_{i:03d}.png'
    if os.path.exists(img_path):
        print(f"Sample {i}:")
        display(Image(filename=img_path, width=800))
        print("\n")

## 10. Download Results

In [ ]:
# Download best model
from google.colab import files

# Zip results
!zip -r results.zip /content/checkpoints /content/results

# Download
files.download('results.zip')

print("✓ Download started!")
print("Contains:")
print("  - Best model checkpoint")
print("  - Predictions CSV")
print("  - Grad-CAM visualizations")

## 11. Save to Google Drive (Recommended)

In [ ]:
# Copy checkpoints and results to Google Drive
!cp -r /content/checkpoints /content/drive/MyDrive/MURA_Results/
!cp -r /content/results /content/drive/MyDrive/MURA_Results/

print("✓ Results saved to Google Drive: MyDrive/MURA_Results/")
print("This ensures you won't lose your work when Colab session ends!")

## Tips for Google Colab

1. **Session Timeout:** Colab disconnects after ~90 minutes of inactivity
   - Keep browser tab active
   - Or use Colab Pro for longer sessions

2. **Save Frequently:**
   - Checkpoints are automatically saved each epoch
   - Copy to Drive periodically: `!cp /content/checkpoints/best.pth /content/drive/MyDrive/`

3. **Resume Training:**
   - If disconnected, rerun setup cells
   - Use: `trainer.train(num_epochs=10, resume_from='/content/checkpoints/latest.pth')`

4. **Free GPU Limits:**
   - ~12 hours per day
   - Consider Colab Pro ($10/month) for unlimited

5. **Dataset Upload:**
   - Upload to Drive once, reuse in all sessions
   - Much faster than re-uploading each time